# Phase B — threshold sweep on the full corpus

Items 5 and 6. Re-runs `EDA.py`'s section 9 sweep on 162k matches instead of
7.4k, and decides whether the motivating anomaly survives the larger sample.

The point is comparability, so the model, features and thresholds are the ones
`EDA.py` used. Three deviations, all strictly-more-correct and none of which
changes the estimator:

- the train/test cut lands on a date boundary instead of mid-day (EDA.py had
  31 matches from one round straddling the split)
- the bootstrap resamples **matches**, not bets — bets on the same fixture are
  not independent and resampling bets understates the interval
- form is recovered with a `side` column instead of `iloc[0::2]`, with an
  equivalence check against the old slicing

Everything else is deliberately unchanged. Walk-forward splits, calibration and
shrinkage belong to Phase C onward; mixing them in here would make it
impossible to tell whether a change in the curve came from more data or from
the changes.


## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

pd.set_option("display.width", 170)

PARQUET = "matches_multiseason.parquet"
THRESHOLDS = [0.00, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15]   # EDA.py's grid
N_BOOT = 2000
SEED = 0

df = pd.read_parquet(PARQUET)
print(f"{len(df):,} matches, {df.Div.nunique()} divisions, "
      f"{df.Date.min().date()} to {df.Date.max().date()}")


162,053 matches, 22 divisions, 2005-07-29 to 2026-08-20


In [2]:
# same analysis set as EDA.py section 9: played matches with opening B365 prices
ODDS = ["B365H", "B365D", "B365A"]
d = df[df.FTR.isin(["H", "D", "A"])].dropna(subset=ODDS + ["FTHG", "FTAG"]).copy()

# dropna misses these: older files write 0 for "no price" instead of leaving the
# cell blank, and 1/0 -> inf -> NaN after normalising. A negative or sub-1 price
# is worse, since it yields a finite but nonsense probability that never errors.
# Decimal odds must exceed 1.
valid = (d[ODDS] > 1.0).all(axis=1)
if (~valid).any():
    print(f"dropping {(~valid).sum():,} matches with invalid odds (<= 1.0):")
    display(d.loc[~valid, ["Season", "Div", "Date", "HomeTeam", "AwayTeam"] + ODDS].head(10))
    print(d.loc[~valid, "Season"].value_counts().sort_index().to_string())
d = d[valid].sort_values("Date").reset_index(drop=True)

s = 1 / d.B365H + 1 / d.B365D + 1 / d.B365A
d["oH"], d["oD"], d["oA"] = (1 / d.B365H) / s, (1 / d.B365D) / s, (1 / d.B365A) / s
d["overround"] = s
y = d.FTR.map({"H": 0, "D": 1, "A": 2}).values.astype(int)

assert np.isfinite(d[["oH", "oD", "oA"]].values).all(), "non-finite de-vigged probabilities"
print(f"\nanalysis set: {len(d):,} matches")
print(f"mean overround: {(d.overround.mean() - 1) * 100:.2f}%")


dropping 10 matches with invalid odds (<= 1.0):


,Season,Div,Date,HomeTeam,AwayTeam,B365H,B365D,B365A
58304,1213,E3,2013-01-12,Chesterfield,Northampton,0.0,3.60,3.25
61358,1213,E1,2013-04-27,Blackpool,Derby,0.0,3.40,3.40
76006,1415,E2,2015-03-28,Sheffield United,Crewe,0.0,5.25,10.00
105612,1819,E1,2019-02-02,Brentford,Blackburn,0.0,3.75,4.20
105669,1819,SC1,2019-02-02,Inverness C,Partick,0.0,3.50,4.50
106156,1819,E2,2019-02-19,Portsmouth,Bristol Rvs,0.0,3.80,4.75
106694,1819,SC3,2019-03-09,Elgin,Clyde,0.0,3.80,1.80
113940,1920,SC1,2020-02-22,Morton,Alloa,0.0,3.60,3.80
117840,2021,E2,2020-12-05,Sunderland,Wigan,0.0,5.00,8.50
118525,2021,SC2,2020-12-26,Clyde,Airdrie Utd,0.0,4.20,2.10


1213    2
1415    1
1819    4
1920    1
2021    2

analysis set: 162,043 matches
mean overround: 6.80%


## 1. Market baseline

Sanity gate before anything is built on top. The de-vigged market should score
a log loss near 1.00 on 1X2; a number far from that means the de-vigging or the
outcome coding is wrong and every later figure would be wrong with it.


In [3]:
def ll(P, yy):
    return log_loss(yy, np.clip(P, 1e-9, 1), labels=[0, 1, 2])


def rps(P, yy):
    # ranked probability score: respects the H < D < A ordering
    Y = np.eye(3)[yy]
    return np.mean(np.sum((np.cumsum(P, 1) - np.cumsum(Y, 1))[:, :2] ** 2, 1) / 2)


Q = d[["oH", "oD", "oA"]].values
print(f"market log loss : {ll(Q, y):.4f}   (want ~1.00)")
print(f"market RPS      : {rps(Q, y):.4f}")
print(f"outcome mix     : H {(y == 0).mean() * 100:.1f}%  "
      f"D {(y == 1).mean() * 100:.1f}%  A {(y == 2).mean() * 100:.1f}%")


market log loss : 1.0056   (want ~1.00)
market RPS      : 0.2047
outcome mix     : H 44.2%  D 26.5%  A 29.3%


In [4]:
# per season, to see whether the market's own quality drifts across the corpus
rows = []
for season, g in d.groupby("Season"):
    yy = g.FTR.map({"H": 0, "D": 1, "A": 2}).values.astype(int)
    rows.append((season, len(g), (g.overround.mean() - 1) * 100,
                 ll(g[["oH", "oD", "oA"]].values, yy), rps(g[["oH", "oD", "oA"]].values, yy)))
era = pd.DataFrame(rows, columns=["Season", "n", "Margin%", "LogLoss", "RPS"])
display(era.round(4))


,Season,n,Margin%,LogLoss,RPS
0,0506,7764,10.9734,1.0128,0.2044
1,0607,7799,10.5400,1.0105,0.2071
2,0708,7803,9.0708,1.0113,0.2055
3,0809,7787,7.5011,1.0114,0.2059
4,0910,7610,6.9471,1.0017,0.2025
5,1011,7734,6.8448,1.0133,0.2063
6,1112,7698,6.5060,1.0058,0.2049
7,1213,7736,6.0290,1.0199,0.2089
8,1314,7796,5.5942,1.0037,0.2052
9,1415,7843,5.3653,1.0060,0.2046


## 2. Favourite-longshot bias

`EDA.py` measured 0.113 priced against 0.076 observed at the long end on one
season — ratio 0.68. Item 16 uses that as the acceptance test for the de-vigging
comparison, so it needs to be re-measured on the full corpus before anything is
built on it.


In [5]:
rows = []
for side, odds, hit in [("H", d.B365H, d.FTR == "H"),
                        ("D", d.B365D, d.FTR == "D"),
                        ("A", d.B365A, d.FTR == "A")]:
    rows.append(pd.DataFrame({"p": (1 / odds) / d.overround, "o": odds,
                              "w": hit.astype(float), "side": side}))
b = pd.concat(rows, ignore_index=True)
b["bin"] = pd.cut(b.p, [0, .05, .10, .20, .30, .40, .50, .60, .70, 1.0])

tbl = b.groupby("bin", observed=True).apply(lambda g: pd.Series({
    "n": len(g), "Predicted": g.p.mean(), "Observed": g.w.mean(),
    "Ratio": g.w.mean() / g.p.mean(),
    "FlatROI%": np.where(g.w > 0, g.o - 1, -1).mean() * 100,
}))
display(tbl.round(3))

lo, hi = b[b.p < 0.15], b[b.p > 0.60]
print(f"longshots  (p<0.15): priced {lo.p.mean():.3f}, won {lo.w.mean():.3f}, "
      f"ratio {lo.w.mean() / lo.p.mean():.2f}")
print(f"favourites (p>0.60): priced {hi.p.mean():.3f}, won {hi.w.mean():.3f}, "
      f"ratio {hi.w.mean() / hi.p.mean():.2f}")
print("EDA.py on one season: longshot ratio 0.68 (0.113 priced, 0.076 won)")


,n,Predicted,Observed,Ratio,FlatROI%
bin,,,,,
"(0.0, 0.05]",1007.0,0.043,0.024,0.558,-47.468
"(0.05, 0.1]",7873.0,0.079,0.061,0.770,-28.477
"(0.1, 0.2]",49627.0,0.160,0.149,0.932,-13.071
"(0.2, 0.3]",199268.0,0.262,0.259,0.985,-7.773
"(0.3, 0.4]",102600.0,0.343,0.343,1.000,-6.345
"(0.4, 0.5]",64420.0,0.447,0.452,1.010,-5.376
"(0.5, 0.6]",35889.0,0.546,0.557,1.021,-4.531
"(0.6, 0.7]",15690.0,0.645,0.668,1.036,-2.946
"(0.7, 1.0]",9755.0,0.765,0.807,1.055,-1.014


longshots  (p<0.15): priced 0.110, won 0.094, ratio 0.86
favourites (p>0.60): priced 0.691, won 0.721, ratio 1.04
EDA.py on one season: longshot ratio 0.68 (0.113 priced, 0.076 won)


## 3. Features

Chronological Elo and 6-match form, exactly as `EDA.py` built them. Ratings are
recorded before the update, so a match never informs its own feature.


In [6]:
elo, K, HFA = {}, 20, 60
eh, ea = np.zeros(len(d)), np.zeros(len(d))

for i, r in enumerate(d.itertuples()):
    kh, ka = (r.Div, r.HomeTeam), (r.Div, r.AwayTeam)
    Rh, Ra = elo.get(kh, 1500.), elo.get(ka, 1500.)
    eh[i], ea[i] = Rh, Ra                       # before the update
    e = 1 / (1 + 10 ** (-((Rh + HFA) - Ra) / 400))
    S = 1.0 if r.FTR == "H" else 0.5 if r.FTR == "D" else 0.0
    m = np.log(max(abs(r.FTHG - r.FTAG), 1) + 1)
    elo[kh], elo[ka] = Rh + K * m * (S - e), Ra - K * m * (S - e)

d["elo_diff"] = (eh + HFA) - ea
print(f"elo_diff: mean {d.elo_diff.mean():.1f}, sd {d.elo_diff.std():.1f}, "
      f"range [{d.elo_diff.min():.0f}, {d.elo_diff.max():.0f}]")


elo_diff: mean 59.9, sd 130.2, range [-525, 660]


In [7]:
# side column rather than iloc[0::2] -- positional recovery depends on the sort
# staying stable and silently swaps home/away if it doesn't
long = pd.concat([
    pd.DataFrame({"i": d.index, "L": d.Div, "T": d.HomeTeam,
                  "gd": d.FTHG - d.FTAG, "side": "H"}),
    pd.DataFrame({"i": d.index, "L": d.Div, "T": d.AwayTeam,
                  "gd": d.FTAG - d.FTHG, "side": "A"}),
]).sort_values("i", kind="stable")

long["f"] = (long.groupby(["L", "T"]).gd
             .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
             .fillna(0))

H = long[long.side == "H"].set_index("i").f
A = long[long.side == "A"].set_index("i").f
d["form_diff"] = (H - A).reindex(d.index).values

# EDA.py's positional version, to confirm the two agree on this corpus
f = long.f.values
old = f[0::2] - f[1::2]
agree = np.allclose(old, d.form_diff.values)
print(f"positional slicing agrees with the side filter: {agree}")
if not agree:
    n = int((~np.isclose(old, d.form_diff.values)).sum())
    print(f"  {n:,} of {len(d):,} matches differ -- EDA.py's form_diff was wrong")


positional slicing agrees with the side filter: True


## 4. Model and split

Same features and model as `EDA.py`. The cut is moved to a date boundary so no
round straddles train and test.


In [8]:
FEATS = ["elo_diff", "form_diff", "oH", "oD", "oA"]

cut = int(len(d) * 0.7)
cut_date = d.Date.iloc[cut]
cut = int((d.Date < cut_date).sum())        # snap to the date boundary
tr, te = np.arange(cut), np.arange(cut, len(d))

X = np.nan_to_num(d[FEATS].values.astype(float))
model = LogisticRegression(max_iter=3000).fit(X[tr], y[tr])

P = model.predict_proba(X[te])
O = d[["B365H", "B365D", "B365A"]].values[te]
yt = y[te]
Qte = d[["oH", "oD", "oA"]].values[te]

print(f"train {cut:,} matches, {d.Date.iloc[0].date()} to {d.Date.iloc[cut - 1].date()}")
print(f"test  {len(te):,} matches, {d.Date.iloc[cut].date()} to {d.Date.iloc[-1].date()}")
print(f"straddling dates: {len(set(d.Date.iloc[tr]) & set(d.Date.iloc[te]))} (want 0)")
print()
print(f"test log loss, market : {ll(Qte, yt):.4f}")
print(f"test log loss, model  : {ll(P, yt):.4f}")
print(f"test RPS,     market : {rps(Qte, yt):.4f}")
print(f"test RPS,     model  : {rps(P, yt):.4f}")


train 113,421 matches, 2005-07-29 to 2020-02-03
test  48,622 matches, 2020-02-04 to 2026-08-20
straddling dates: 0 (want 0)

test log loss, market : 1.0018
test log loss, model  : 1.0014
test RPS,     market : 0.2039
test RPS,     model  : 0.2038


## 5. Threshold sweep

Two variants. `all` takes every outcome clearing the threshold, which is what
`EDA.py` did and lets one fixture contribute several bets. `one` keeps only the
highest-EV outcome per match, which is the coherent strategy. Reporting both
shows whether the pattern depends on that choice.

The bootstrap resamples matches and reuses the same resampled fixtures across
every threshold, so the differences between thresholds are paired.


In [9]:
EV = P * O - 1


def select(th, one_per_match):
    sel = EV >= th
    if not one_per_match:
        return np.where(sel)
    best = np.where(sel.any(1))[0]
    if len(best) == 0:
        return np.array([], int), np.array([], int)
    cols = np.argmax(np.where(sel, EV, -np.inf)[best], axis=1)
    return best, cols


def sweep(one_per_match, n_boot=N_BOOT, seed=SEED):
    n = len(te)
    S = np.zeros((len(THRESHOLDS), n))     # per-match P&L sum, per threshold
    C = np.zeros((len(THRESHOLDS), n))     # per-match bet count
    rows = []

    for k, th in enumerate(THRESHOLDS):
        ri, ci = select(th, one_per_match)
        if len(ri) == 0:
            rows.append((th, 0, np.nan, np.nan))
            continue
        won = ci == yt[ri]
        pnl = np.where(won, O[ri, ci] - 1, -1.0)
        S[k] = np.bincount(ri, weights=pnl, minlength=n)
        C[k] = np.bincount(ri, minlength=n)
        rows.append((th, len(ri), won.mean() * 100, pnl.mean() * 100))

    # paired bootstrap over fixtures
    rng = np.random.default_rng(seed)
    boot = np.empty((n_boot, len(THRESHOLDS)))
    for bnum in range(n_boot):
        idx = rng.integers(0, n, n)
        num, den = S[:, idx].sum(1), C[:, idx].sum(1)
        boot[bnum] = np.where(den > 0, num / np.maximum(den, 1) * 100, np.nan)

    out = pd.DataFrame(rows, columns=["tau", "Bets", "HitRate%", "ROI%"])
    out["CIlow"] = np.nanpercentile(boot, 2.5, axis=0)
    out["CIhigh"] = np.nanpercentile(boot, 97.5, axis=0)
    out["P(ROI<=0)"] = np.nanmean(boot <= 0, axis=0)
    return out.round(3), boot


sweep_all, boot_all = sweep(one_per_match=False)
sweep_one, boot_one = sweep(one_per_match=True)

print("all qualifying outcomes (EDA.py's rule):")
display(sweep_all)
print("one bet per match:")
display(sweep_one)


all qualifying outcomes (EDA.py's rule):


,tau,Bets,HitRate%,ROI%,CIlow,CIhigh,P(ROI<=0)
0,0.000,4632,33.484,-10.815,-16.505,-4.761,1.000
1,0.010,2651,21.011,-12.912,-22.071,-3.102,0.994
2,0.020,1962,11.468,-18.292,-30.025,-6.272,0.998
3,0.030,1706,8.910,-21.237,-33.695,-7.865,0.998
4,0.050,1440,7.569,-24.965,-38.899,-10.518,1.000
5,0.075,1197,7.101,-28.154,-42.809,-11.856,1.000
6,0.100,984,6.809,-29.522,-45.630,-12.316,1.000
7,0.150,664,6.325,-28.163,-49.167,-6.115,0.994


one bet per match:


,tau,Bets,HitRate%,ROI%,CIlow,CIhigh,P(ROI<=0)
0,0.000,4594,33.457,-11.470,-17.008,-5.332,1.000
1,0.010,2642,20.931,-14.302,-23.413,-4.492,0.997
2,0.020,1956,11.350,-20.239,-31.610,-8.274,1.000
3,0.030,1701,8.818,-22.770,-35.499,-9.558,0.999
4,0.050,1435,7.456,-26.794,-40.134,-12.196,1.000
5,0.075,1194,7.035,-29.229,-43.738,-13.284,1.000
6,0.100,984,6.809,-29.522,-45.630,-12.316,1.000
7,0.150,664,6.325,-28.163,-49.167,-6.115,0.994


## 6. The decision (item 6)

Two overlapping intervals do not settle whether the curve declines, and neither
does comparing the two endpoints — the tightest threshold carries the fewest
bets and the widest interval, so an endpoint test hangs on the noisiest point
in the sweep.

H1 says ROI is *decreasing* in tau. That is a monotonicity claim, not a
linearity one, so the primary test is the bootstrapped **Spearman correlation**
between tau and ROI across every usable threshold — shape-agnostic, and it does
not care that the curve is concave. A linear slope is reported alongside as a
magnitude, but it is the wrong test on its own: fitting a straight line through
a concave curve understates the decline, and a noisy upturn at the last
threshold inflates its variance enough to drag the interval over zero.

The per-threshold table then shows *where* the gap against tau = 0 first
excludes zero, which is more useful than any single number.

rho significantly negative → the decline is real and the specification stands.
Interval spanning zero → the anomaly was a single-season artefact and H1 needs
reframing before anything else is written.


In [10]:
MIN_BETS = 200          # thresholds thinner than this are not worth a verdict


def spearman(x, y):
    rx = x.argsort().argsort() + 1.0
    ry = y.argsort().argsort() + 1.0
    a, b = rx - rx.mean(), ry - ry.mean()
    return (a @ b) / np.sqrt((a ** 2).sum() * (b ** 2).sum())


def decision(sweep_df, boot, label):
    k = sweep_df.index[sweep_df.Bets >= MIN_BETS].values
    if len(k) < 3:
        print(f"{label}: fewer than three thresholds clear {MIN_BETS} bets")
        return
    taus = sweep_df.tau.values[k]
    roi = sweep_df["ROI%"].values[k]
    B = boot[:, k]
    B = B[~np.isnan(B).any(1)]

    # Spearman: H1 claims ROI falls with tau, not that it falls linearly.
    # Ordinal ranks are fine here -- ties among bootstrap ROIs are vanishing.
    def ranks(M):
        order = M.argsort(1)
        r = np.empty_like(order)
        np.put_along_axis(r, order, np.arange(M.shape[1]), axis=1)
        return r + 1.0

    rb = ranks(B) - (len(taus) + 1) / 2
    xr = (taus.argsort().argsort() + 1.0) - (len(taus) + 1) / 2
    rho = (rb @ xr) / np.sqrt((rb ** 2).sum(1) * (xr ** 2).sum())
    r_lo, r_hi = np.percentile(rho, [2.5, 97.5])

    # linear slope, as a magnitude only -- shape-sensitive, see the markdown
    x = taus - taus.mean()
    slope = (B - B.mean(1, keepdims=True)) @ x / (x @ x)
    s_lo, s_hi = np.percentile(slope, [2.5, 97.5])

    print(f"--- {label} ---")
    print(f"thresholds used : {', '.join(f'{t:.3f}' for t in taus)}")
    print(f"Spearman rho    : {spearman(taus, roi):+.3f}   "
          f"95% CI [{r_lo:+.3f}, {r_hi:+.3f}]   P(rho<0) = {(rho < 0).mean():.3f}")
    print(f"linear slope    : {np.polyfit(taus, roi, 1)[0] / 100:+.2f} ROI% per 1pp   "
          f"95% CI [{s_lo / 100:+.2f}, {s_hi / 100:+.2f}]   (magnitude only)")
    print(f"monotone to the minimum: "
          f"{bool(np.all(np.diff(roi[:int(np.argmin(roi)) + 1]) < 0))}")

    if r_hi < 0:
        print("=> decline confirmed at this sample size")
    elif r_lo > 0:
        print("=> return RISES with selectivity -- the opposite of H1")
    else:
        print("=> interval spans zero: no decline established. H1 needs reframing.")

    # where the gap against tau=0 first excludes zero
    gap = B[:, [0]] - B
    rows = []
    for j in range(1, len(taus)):
        l, h = np.percentile(gap[:, j], [2.5, 97.5])
        rows.append((taus[j], roi[0] - roi[j], l, h, "yes" if l > 0 else "no"))
    display(pd.DataFrame(
        rows, columns=["tau", "ROI(0)-ROI(tau)", "CIlow", "CIhigh", "excludes 0"]
    ).round(2))
    print()


decision(sweep_all, boot_all, "all qualifying outcomes")
decision(sweep_one, boot_one, "one bet per match")


--- all qualifying outcomes ---
thresholds used : 0.000, 0.010, 0.020, 0.030, 0.050, 0.075, 0.100, 0.150
Spearman rho    : -0.976   95% CI [-1.000, -0.143]   P(rho<0) = 0.989
linear slope    : -1.20 ROI% per 1pp   95% CI [-2.41, +0.12]   (magnitude only)
monotone to the minimum: True
=> decline confirmed at this sample size


,tau,ROI(0)-ROI(tau),CIlow,CIhigh,excludes 0
0,0.01,2.10,-2.53,6.70,no
1,0.02,7.48,-0.08,14.71,no
2,0.03,10.42,1.52,18.97,yes
3,0.05,14.15,3.78,24.79,yes
4,0.08,17.34,4.60,28.88,yes
5,0.10,18.71,4.03,32.99,yes
6,0.15,17.35,-2.98,36.62,no



--- one bet per match ---
thresholds used : 0.000, 0.010, 0.020, 0.030, 0.050, 0.075, 0.100, 0.150
Spearman rho    : -0.929   95% CI [-1.000, -0.095]   P(rho<0) = 0.982
linear slope    : -1.10 ROI% per 1pp   95% CI [-2.30, +0.21]   (magnitude only)
monotone to the minimum: True
=> decline confirmed at this sample size


,tau,ROI(0)-ROI(tau),CIlow,CIhigh,excludes 0
0,0.01,2.83,-1.76,7.22,no
1,0.02,8.77,1.23,15.85,yes
2,0.03,11.30,2.43,19.76,yes
3,0.05,15.32,4.70,25.57,yes
4,0.08,17.76,4.82,29.31,yes
5,0.10,18.05,3.40,32.27,yes
6,0.15,16.69,-3.39,36.02,no


## 7. Does it hold in both halves?

If the decline is a property of the selection rule it should appear in both
halves of the corpus. If it only appears in one, it is an era effect and the
single-season result was picking up whichever era it sat in.


In [11]:
mid = len(te) // 2
halves = {"first half of test": np.arange(mid), "second half of test": np.arange(mid, len(te))}

rows = []
for name, sub in halves.items():
    for th in (0.0, 0.05, 0.10):
        sel = EV[sub] >= th
        ri, ci = np.where(sel)
        if len(ri) < 50:
            rows.append((name, th, len(ri), np.nan))
            continue
        won = ci == yt[sub][ri]
        pnl = np.where(won, O[sub][ri, ci] - 1, -1.0)
        rows.append((name, th, len(ri), pnl.mean() * 100))

display(pd.DataFrame(rows, columns=["Half", "tau", "Bets", "ROI%"]).round(2))


,Half,tau,Bets,ROI%
0,first half of test,0.00,2266,-10.25
1,first half of test,0.05,766,-19.84
2,first half of test,0.10,525,-21.71
3,second half of test,0.00,2366,-11.35
4,second half of test,0.05,674,-30.79
5,second half of test,0.10,459,-38.45


## 8. Shrinkage weight (H3)

`EDA.py` estimated w on one season and got a wide interval containing 1, which
it called "not usable". Same regression on the full test set — this is the first
number in the project that speaks to H3 directly.


In [12]:
dev = P[:, 0] - Qte[:, 0]                    # model minus market, home
res = (yt == 0).astype(float) - Qte[:, 0]    # truth minus market, home

w, b0 = np.polyfit(dev, res, 1)
n = len(dev)
se = np.sqrt(np.sum((res - (w * dev + b0)) ** 2) / (n - 2) / np.sum((dev - dev.mean()) ** 2))

print(f"n                : {n:,}")
print(f"estimated w      : {w:+.4f}")
print(f"standard error   : {se:.4f}")
print(f"t                : {w / se:+.2f}")
print(f"95% CI           : [{w - 1.96 * se:+.3f}, {w + 1.96 * se:+.3f}]")
print(f"mean |deviation| : {np.abs(dev).mean():.4f}   (EDA.py: 0.019)")
print()
print("w=0: the model's disagreements are noise, shrink them away entirely.")
print("w=1: take them at face value.")


n                : 48,622
estimated w      : +0.8609
standard error   : 0.1338
t                : +6.43
95% CI           : [+0.599, +1.123]
mean |deviation| : 0.0132   (EDA.py: 0.019)

w=0: the model's disagreements are noise, shrink them away entirely.
w=1: take them at face value.


## 9. The figure

In [13]:
plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 9,
    "axes.edgecolor": "#5A6472", "axes.labelcolor": "#1A2332",
    "text.color": "#1A2332", "xtick.color": "#5A6472", "ytick.color": "#5A6472",
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.grid": True,
    "grid.color": "#E4E9ED", "grid.linewidth": 0.7,
})
ACC, RED = "#2D6A8F", "#9B3D3D"

fig, ax = plt.subplots(figsize=(7.5, 4.6))
for sw, colour, label in [(sweep_all, ACC, "all qualifying outcomes"),
                          (sweep_one, RED, "one bet per match")]:
    ok = sw[sw.Bets >= MIN_BETS]
    ax.fill_between(ok.tau * 100, ok.CIlow, ok.CIhigh, color=colour, alpha=.12)
    ax.plot(ok.tau * 100, ok["ROI%"], "o-", c=colour, lw=1.8, ms=5, label=label)
    for _, r in ok.iterrows():
        ax.annotate(f"n={int(r.Bets):,}", (r.tau * 100, r["ROI%"]),
                    textcoords="offset points", xytext=(0, -14),
                    fontsize=6.5, ha="center", color="#5A6472")

ax.axhline(0, c="#9AA5AF", lw=1, ls="--")
ax.set_xlabel("Selection threshold  $\\tau$  (%)")
ax.set_ylabel("Return on investment (%)")
ax.set_title(f"ROI against selectivity, {len(te):,} test matches",
             fontsize=10, weight="bold", loc="left")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig("phase_b_threshold_sweep.png", dpi=190, bbox_inches="tight")
print("written to phase_b_threshold_sweep.png")
plt.show()


written to phase_b_threshold_sweep.png


/var/folders/q9/lc4q36s51dv4pxz0gn3sssq00000gn/T/ipykernel_10280/2111491169.py:31: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Next

Read section 6 before writing anything else.

- **Decline confirmed** → the specification stands. Build the harness (items
  7-11) next; nothing downstream is trustworthy until the metrics, the
  walk-forward splitter and the leakage suite exist.
- **Interval spans zero** → H1 as stated is not supported at this sample size.
  The simulation study (items 12-15) still stands on its own since it needs no
  football data, and the framing shifts from "the anomaly exists and here is
  the correction" to "here is when the correction matters, tested where the
  noise is known".
